# Design Principles in Practice: A Live Code Review

**Course:** Software Engineering  
**Topic:** Deep Modules, Testability, and the TDD Question

---

We are going to review a set of implementation issues for a real project: a voice agent built in pure Python using asyncio.

The issues were written with specific design goals in mind:

> *"Deep modules with simple interfaces that hide complexities. Digestible tasks that touch as many layers of the system as possible. Every completed issue must be testable."*

Our job today: **inspect whether the issues actually achieve those goals.**

By the end of this session you will be able to:
- Recognize deep modules vs. shallow modules in real code
- Explain the difference between TDD and Design for Testability
- Identify when a task is not actually testable even if it looks like it is
- Write a well-formed issue that embeds testability from the start

---

## Part 1 — The Design Framework

### 1.1 Deep Modules

The central idea comes from John Ousterhout's *A Philosophy of Software Design* (2018).

> *"The best modules are those that provide powerful functionality yet have simple interfaces. I use the term deep to describe such modules."*

A **deep module** has:
- A **narrow interface** — few methods, few parameters, clear contract
- A **complex implementation** — lots of work happens inside, invisible to the caller

A **shallow module** has an interface almost as complex as its implementation. It adds no value — the caller might as well do the work themselves.

```
┌─────────────────────────────────────────────────────┐
│  DEEP MODULE                  SHALLOW MODULE         │
│                                                      │
│  ┌────────────────┐           ┌───────────────────┐  │
│  │ interface: 1-2 │           │ interface: 8 args │  │
│  │   methods      │           │ 12 methods        │  │
│  ├────────────────┤           ├───────────────────┤  │
│  │                │           │                   │  │
│  │  complex impl  │           │  simple impl      │  │
│  │  WebSocket     │           │  pass-through     │  │
│  │  reconnect     │           │  mostly           │  │
│  │  buffering     │           │                   │  │
│  │  JSON parsing  │           │                   │  │
│  │  error retry   │           │                   │  │
│  └────────────────┘           └───────────────────┘  │
└─────────────────────────────────────────────────────┘
```

In [ ]:
# Example of a deep module: the STTProvider Protocol from our voice agent
#
# The interface is two methods. That is all callers ever see.
# Behind it: WebSocket lifecycle, reconnect logic, partial vs. final
# transcript framing, API auth, audio format negotiation.

from typing import Protocol, AsyncIterator

class STTProvider(Protocol):
    async def transcribe_stream(
        self,
        audio_stream: AsyncIterator[bytes],
    ) -> AsyncIterator[tuple[str, bool]]:  # (text, is_final)
        ...

    async def close(self) -> None:
        ...

# The caller (the STT processor stage) does exactly this:
#
#   async for text, is_final in provider.transcribe_stream(audio):
#       await output_queue.put(TranscriptionFrame(text=text, is_final=is_final))
#
# It has no idea whether the provider is Deepgram, faster-whisper, or a mock.
# That is information hiding in action.

print("STTProvider interface: 2 methods")
print("Behind it: WebSocket, auth, reconnect, partial/final framing...")
print("The caller sees NONE of that complexity.")

### 1.2 Information Hiding

Information hiding is the mechanism that produces deep modules. Each module **hides a design decision** — something that could change, or something that is complex to understand.

In our voice agent, every provider hides a different decision:

| Module | Hidden decision |
|---|---|
| `SileroVAD` | Which ML model, tensor conversion, silence threshold tuning |
| `DeepgramSTT` | WebSocket protocol, reconnect strategy, partial vs. final transcripts |
| `GroqLLM` | Streaming token protocol, context formatting, OpenAI-compatible API |
| `CartesiaTTS` | HTTP streaming, audio chunking, PCM encoding |
| `ConversationContext` | Sliding window arithmetic, message format |
| `SentenceAggregator` | Punctuation parsing, partial buffer management |

The **processor stages** (VAD, STT, LLM, TTS) interact only with the Protocol interface. They never know which provider is underneath.

### 1.3 Vertical Slices

Each issue in this project was designed to touch **multiple system layers** simultaneously. The goal: every completed issue produces something end-to-end observable, not just a utility floating in isolation.

The system has these layers:

```
config.py         ← env var loading
frames.py         ← data types that flow through the pipeline
processors/       ← computation stages (VAD, STT, LLM, TTS)
pipeline.py       ← queue wiring + task orchestration
utils/            ← pure functions (sentence splitting, timing)
tests/            ← automated verification
```

A good issue touches at least 3-4 of these layers. We will inspect whether the actual issues live up to this.

---

## Part 2 — TDD vs. Design for Testability

### The question students always ask: "Is this TDD?"

**Short answer: No. But understanding why matters more than the label.**

---

### 2.1 What TDD actually is

Test-Driven Development is a **development process**:

```
 1. RED    → Write a test that fails (because the feature doesn't exist yet)
 2. GREEN  → Write the minimum code to make the test pass
 3. REFACTOR → Clean up, without breaking the test
 4. Repeat
```

The test comes **before** the implementation. Always. That is the defining property.

TDD is a discipline about **when** you write tests and **how** that shapes your implementation decisions.

---

### 2.2 What Design for Testability is

Design for Testability is a **design property**:

> A module is testable if you can inject its inputs, observe its outputs, and replace its dependencies — without touching real hardware, real APIs, or real filesystems.

The Provider Protocol pattern in our project is *Design for Testability*. Because each processor depends on an interface (not a concrete class), you can inject a mock and test the processor logic in isolation.

---

### 2.3 How they relate

TDD **forces** good design for testability — if you write the test first, you are forced to think about the interface before the implementation. Modules that are hard to test are hard to write tests for first, so TDD practitioners naturally end up with narrow, injectable interfaces.

But you can have good Design for Testability without doing TDD. The Provider Protocol pattern is evidence of that: the interfaces are mockable by design, even though the tests were not written first.

```
TDD          →  always produces testable design (process enforces it)
Design for   →  testable design, process is up to you
Testability
```

---

### 2.4 What is actually missing from these issues

The issues have good *design* for testability — the interfaces are right. What they lack is the *instruction to test*.

Issues 01 through 09 have no test requirements. All testing is deferred to Issue 10.

**This means:** a student who follows the issues exactly will build 9 features without writing a single automated test. Then Issue 10 asks them to write tests for all of it at once.

That is the wrong lesson. Let's see this in the actual issues.

---

## Part 3 — Live Issue Inspection

### 3.1 Overview: how many layers does each issue touch?

| Issue | Title | config | frames | processor | pipeline | utils | tests | Layers |
|---|---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| 01 | Scaffold + echo loop | ✓ | ✓ | ✓ | ✓ | | ✗ | 4 |
| 02 | VAD stage | | ✓ | ✓ | ✓ | | ✗ | 3 |
| 03 | STT stage | ✓ | ✓ | ✓ | ✓ | | ✗ | 4 |
| 04 | LLM stage | ✓ | ✓ | ✓ | ✓ | ✓ | ✗ | 5+1 |
| 05 | TTS + full loop | ✓ | ✓ | ✓ | ✓ | | ✗ | 4 |
| 06 | CancelFrame | | ✓ | ✓ | ✓ | ✓ | ✗ | 4 |
| 07 | ErrorFrame | | ✓ | ✓ | ✓ | | ✗ | 3 |
| 08 | Alt providers | | | ✓ | | | ✗ | **1** |
| 09 | Latency tracking | | | ✓ | ✓ | ✓ | ✗ | 3 |
| 10 | Unit test suite | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | 6 |
| 11 | Integration tests | | | ✓ | | | ✓ | 2 |

**Notice:** The `tests` column is empty for every issue except 10 and 11. And Issue 08 only touches 1 layer. Let's dig into the three structural problems.

### 3.2 Problem 1 — Testing deferred to Issue 10

Read the first sentence of Issue 10:

> *"Write a comprehensive unit test suite covering all deep modules."*

**The word "write" tells the whole story.** Students have been building for 9 issues. Now they are asked to write tests for all of it.

This is the **retrofitting problem**. When tests come after implementation:

1. Students write code optimized for running, not for testing
2. Dependencies get tangled — hard to inject mocks later
3. Testing feels like extra work, not part of the workflow
4. No feedback loop: a bug introduced in Issue 03 isn't caught until Issue 10

Compare this to the stated goal:
> *"Every completed issue must be testable."*

**Question for discussion:** Does "every completed issue must be testable" mean the issue *can* be tested, or that tests *must be written* as part of the issue? Does the current structure satisfy even the weaker reading?

In [ ]:
# Let's look at what Issue 10 asks students to test — after 9 issues without tests.
# This is the list from the actual issue file:

things_to_test_in_issue_10 = [
    "frames.py — instantiation, field defaults",
    "conversation.py — sliding window, turn accumulation, system prompt",
    "utils/sentence_aggregator.py — boundary detection, partial accumulation, CancelFrame flush",
    "processors/vad/ — speaking state frame emission",
    "processors/stt/ — TranscriptionFrame emission, partial vs final",
    "processors/llm/ — LLMResponseFrame streaming, ConversationContext updates",
    "processors/tts/ — TTSAudioFrame chunks, CancelFrame buffer flush",
    "pipeline.py — end-to-end frame flow, CancelFrame propagation",
]

# Some of these could have been tested immediately after being written:
testable_immediately = [
    "frames.py",           # pure dataclasses, zero dependencies
    "conversation.py",     # pure Python, no I/O
    "sentence_aggregator", # pure Python, no I/O
]

# If SentenceAggregator has a bug, when will the student find out?
# Answer: Issue 10 — after implementing Issues 04, 05, 06, 07, 08, 09.

issues_between_04_and_10 = 6
print(f"A bug in SentenceAggregator (written in Issue 04) can hide for {issues_between_04_and_10} issues.")
print("That is a 6-issue feedback loop.")

### 3.3 Problem 2 — Issue 04 is too broad

Read what Issue 04 asks students to build:

- `LLMResponseFrame` (new frame type)
- `LLMProvider` Protocol
- `GroqLLM` implementation (streaming, API integration)
- `ConversationContext` with sliding window
- `SentenceAggregator` utility
- Pipeline wiring update
- `.env.example` update

That is **three independent, testable concepts** bundled into one issue:

```
 ConversationContext   →  pure Python, no dependencies
 SentenceAggregator   →  pure Python, no dependencies
 LLM processor        →  depends on both of the above + Groq API
```

The first two are the simplest things in the entire system to test. They have no dependencies. They are perfect exercises for writing tests.

**If they were their own sub-task**, students would:
1. Implement `SentenceAggregator`
2. Immediately write `tests/test_sentence_aggregator.py`
3. Run the tests, get green, move on
4. Build the LLM processor knowing the utility it depends on is correct

Instead, all three land at once. Students get the LLM processor running and never think about testing `SentenceAggregator` until Issue 10.

In [ ]:
# SentenceAggregator is a pure function — the easiest thing to test in the system.
# Here is what a test for it looks like. Notice: no asyncio, no queues, no mocks.

class SentenceAggregator:
    """Buffers LLM tokens and yields complete sentences."""

    def __init__(self):
        self._buffer = ""

    def push(self, token: str) -> list[str]:
        self._buffer += token
        sentences = []
        while True:
            for i, ch in enumerate(self._buffer):
                if ch in ".!?" and (i + 1 >= len(self._buffer) or self._buffer[i + 1] == " "):
                    sentences.append(self._buffer[: i + 1].strip())
                    self._buffer = self._buffer[i + 1 :].lstrip()
                    break
            else:
                break
        return sentences

    def flush(self) -> list[str]:
        result = [self._buffer.strip()] if self._buffer.strip() else []
        self._buffer = ""
        return result


# A test for this — written immediately after the module, not 6 issues later:
def test_single_sentence():
    agg = SentenceAggregator()
    assert agg.push("Hello") == []
    assert agg.push(", world") == []
    assert agg.push(".") == ["Hello, world."]

def test_multi_sentence_in_one_push():
    agg = SentenceAggregator()
    result = agg.push("First sentence. Second sentence.")
    assert result == ["First sentence.", "Second sentence."]

def test_flush_incomplete_buffer():
    agg = SentenceAggregator()
    agg.push("Incomplete")
    assert agg.flush() == ["Incomplete"]

test_single_sentence()
test_multi_sentence_in_one_push()
test_flush_incomplete_buffer()
print("All SentenceAggregator tests pass.")
print("This took 2 minutes to write and could have been in Issue 04.")

### 3.4 Problem 3 — Issue 08 only touches one layer

The goal was: *"digestible tasks that touch the most quantity possible of the layers of the system."*

Issue 08 asks students to implement `FasterWhisperSTT` and `PiperTTS` — two alternative providers.

**Layers touched:**
- `processors/stt/faster_whisper.py` ✓
- `processors/tts/piper.py` ✓

That is it. No changes to `config.py`, no changes to `pipeline.py`, no tests.

But the **design claim** of the issue is that swapping providers requires only a config change. If we never update `config.py` to support `STT_PROVIDER=faster_whisper`, we never actually verify that claim.

A richer Issue 08 would:
1. Add `stt_provider` and `tts_provider` fields to `config.py`
2. Update `pipeline.py` to instantiate the correct provider from config
3. Require tests that verify both providers satisfy the Protocol

That makes it a 4-layer issue and **proves** the Protocol abstraction works end-to-end.

---

## Part 4 — What a Well-Formed Issue Looks Like

### 4.1 Issue 02 — Before (current state)

```markdown
# 02 — VAD stage — detect speaking state

## What to build
Add a Voice Activity Detection stage...

## Acceptance criteria
- [ ] Speaking and silence transitions are logged to stdout during a live session
- [ ] UserStartedSpeakingFrame is emitted within ~32ms of speech onset
- [ ] UserStoppedSpeakingFrame is emitted reliably after a short silence
- [ ] VADProvider Protocol is defined in base.py with a clear interface
- [ ] Swapping the VAD implementation requires only changing the injected provider object
- [ ] AudioRawFrames are forwarded downstream unchanged
- [ ] EndFrame still propagates cleanly through the VAD stage

## Blocked by
- #01 — Project scaffold + silent echo loop
```

**What is testable?** Only manually — "speaking transitions are logged to stdout" requires a microphone.

The Protocol is defined but never used for testing within this issue.

---

### 4.2 Issue 02 — After (with test section)

```markdown
# 02 — VAD stage — detect speaking state

## What to build
[same as before]

## Acceptance criteria
- [ ] Speaking and silence transitions are logged to stdout during a live session
- [ ] UserStartedSpeakingFrame is emitted within ~32ms of speech onset
- [ ] UserStoppedSpeakingFrame is emitted reliably after a short silence
- [ ] VADProvider Protocol is defined in base.py with a clear interface
- [ ] Swapping the VAD implementation requires only changing the injected provider object
- [ ] AudioRawFrames are forwarded downstream unchanged
- [ ] EndFrame still propagates cleanly through the VAD stage

## Tests to write (tests/test_vad_processor.py)

Define MockVADProvider implementing the VADProvider Protocol.
It takes a list of booleans at construction and returns them in sequence.

- [ ] MockVADProvider defined and satisfies the VADProvider Protocol
- [ ] Test: sequence of speaking=True frames → UserStartedSpeakingFrame emitted once
- [ ] Test: speaking=True followed by speaking=False → UserStoppedSpeakingFrame emitted
- [ ] Test: AudioRawFrames pass through the queue regardless of VAD state
- [ ] Test: EndFrame propagates through the VAD processor
- [ ] make test passes with no API keys, no microphone

## Blocked by
- #01 — Project scaffold + silent echo loop
```

**What changed?**
- Students define `MockVADProvider` as part of this issue — this is the first time they see how the Protocol creates a test seam
- Every acceptance criterion in the "Tests" section is automatically verifiable
- The issue teaches two things at once: the VAD design AND how to use Protocols for testing

In [ ]:
# Here is what the MockVADProvider and its test look like in practice.
# This is the code students would write as part of Issue 02 — not Issue 10.

import asyncio
from dataclasses import dataclass

# --- frames.py (already exists from Issue 01/02) ---
@dataclass
class AudioRawFrame:
    audio: bytes
    sample_rate: int
    num_channels: int

@dataclass
class UserStartedSpeakingFrame:
    pass

@dataclass
class UserStoppedSpeakingFrame:
    pass

@dataclass
class EndFrame:
    pass

# --- processors/vad/base.py ---
class VADProvider(Protocol):
    def is_speech(self, audio: bytes) -> bool:
        ...

# --- MockVADProvider (defined in tests/test_vad_processor.py) ---
class MockVADProvider:
    """Returns a predetermined sequence of True/False values."""
    def __init__(self, sequence: list[bool]):
        self._sequence = iter(sequence)

    def is_speech(self, audio: bytes) -> bool:
        return next(self._sequence, False)

# --- The VAD processor stage (simplified) ---
async def vad_processor(input_queue: asyncio.Queue, output_queue: asyncio.Queue, provider):
    was_speaking = False
    async for frame in _queue_iter(input_queue):
        if isinstance(frame, EndFrame):
            await output_queue.put(frame)
            return
        if isinstance(frame, AudioRawFrame):
            await output_queue.put(frame)  # always forward
            is_speaking = provider.is_speech(frame.audio)
            if is_speaking and not was_speaking:
                await output_queue.put(UserStartedSpeakingFrame())
            elif not is_speaking and was_speaking:
                await output_queue.put(UserStoppedSpeakingFrame())
            was_speaking = is_speaking

async def _queue_iter(q: asyncio.Queue):
    while True:
        item = await q.get()
        yield item

# --- The test ---
async def test_vad_detects_speaking_transition():
    input_q: asyncio.Queue = asyncio.Queue()
    output_q: asyncio.Queue = asyncio.Queue()

    # Mock: first frame is silence, next two are speech, then silence again
    mock = MockVADProvider(sequence=[False, True, True, False])
    frame = AudioRawFrame(audio=b"\x00" * 512, sample_rate=16000, num_channels=1)

    await input_q.put(frame)  # silence
    await input_q.put(frame)  # speech starts
    await input_q.put(frame)  # still speaking
    await input_q.put(frame)  # silence again
    await input_q.put(EndFrame())

    await vad_processor(input_q, output_q, mock)

    outputs = []
    while not output_q.empty():
        outputs.append(output_q.get_nowait())

    types = [type(f).__name__ for f in outputs]
    assert "UserStartedSpeakingFrame" in types
    assert "UserStoppedSpeakingFrame" in types
    assert types.count("UserStartedSpeakingFrame") == 1, "Should only emit once on transition"
    assert types[-1] == "EndFrame", "EndFrame must propagate"
    print("Output frame sequence:", types)
    print("Test passed.")

await test_vad_detects_speaking_transition()

### 4.3 The core principle

> **If something is hard to test, the interface is probably wrong.**

This is the practical consequence of everything above. When you try to write a test and find yourself spinning up real hardware, making real network calls, or wiring together six modules just to test one thing — that is the design telling you something.

The Provider Protocol pattern in this project exists precisely because it makes each stage independently testable. If students only see that in Issue 10, they see it as a testing tool. If they use it in every issue from Issue 02 onward, they understand it as a *design* tool that happens to make testing free.

---

## Summary

| What we found | Design principle violated |
|---|---|
| Tests deferred to Issue 10 | "Every completed issue must be testable" — not enforced |
| Issue 04 bundles 3 independent concepts | Digestible tasks — not achieved for the largest issue |
| Issue 08 touches 1 layer | "Touch as many layers as possible" — not achieved |
| No MockProviders defined before Issue 10 | Protocol as a test seam — taught too late |

**The framework is correct. The issues don't fully implement it.**

The improved issues add a **"Tests to write"** section to every issue (01-09), split Issue 04's pure utilities into an explicit sub-task, and extend Issue 08 to include `config.py` and `pipeline.py` changes.

Those changes are in the `improved-issues` branch.